# Simplex Tabela

In [8]:
# simplex_tableau_frac_noslack.py
from fractions import Fraction

def to_frac_matrix(mat):
    return [[Fraction(x) for x in row] for row in mat]

def simplex_tableau_min_frac(A, b, c, max_iter=50):
    """
    Simplex-tabela (minimização) JÁ na forma-padrão.
    NÃO adiciona novas folgas: considera que as últimas m colunas de A
    são a base inicial.
    """
    # ─── preparação ───
    m, n = len(A), len(A[0])          # m eq., n variáveis (incluindo folgas)
    A = to_frac_matrix(A)
    b = [Fraction(x) for x in b]
    c = [Fraction(x) for x in c]

    # ★ tableau (m+1) × (n+1) – NÃO há colunas extras de s1,s2,s3
    T = [[Fraction(0) for _ in range(n + 1)] for _ in range(m+1)]

    # linha-custo
    for j in range(n):
        T[0][j] = -c[j]

    # restrições
    for i in range(m):
        T[i+1][:n] = A[i]
        T[i+1][-1] = b[i]

    var  = [f"x{j+1}" for j in range(n)]      # ★ só x1…xn
    base = var[-m:]                           # ★ últimas m = folgas prontas

    # ---- impressão opcional ----
    def print_tableau(it_):
        head = ["  "] + var + ["LD"]
        print(f"\n── Tabela – iteração {it_} ──")
        print(" | ".join(f"{h:>8}" for h in head))
        print("-" * (12*(len(var)+2)))
        for i, row in enumerate(T):
            tag = "z" if i == 0 else base[i-1]
            print(f"{tag:>2} | " + " | ".join(f"{str(x):>8}" for x in row))

    print_tableau(0)

    # ---- loop simplex ----
    # ───────── Loop Simplex ─────────
    for it in range(1, max_iter+1):

        # 1. escolhe coluna que entra  (PASSO b)
        enter_col, best = None, Fraction(0)
        for j in range(n):                       # percorre APENAS colunas x1…xn
            if T[0][j] > 0 and T[0][j] > best:
                best, enter_col = T[0][j], j

        if enter_col is None:                    # condição de ótimalidade
            print("\n*** ÓTIMO ALCANÇADO ***")
            print("\n⇒ Solução ótima (frações):")
            for j, val in enumerate(T[0][:-1], 1):
                print(f"x{j} = {val}")
            print("z* =", T[0][-1])
            break

        # ─── (IMPRESSÃO 1)  k e y_k ───────────────────────────────
        yk = [T[i][enter_col] for i in range(1, m+1)]      # coluna k nas linhas 1..m
        print(f"\n♦ Iteração {it}")
        print(f"  k  (entra) : {var[enter_col]}")          # nome da variável que entra
        print(f"  y_k        : {[str(v) for v in yk]}")


        # 2. escolhe linha que sai  (PASSO c)
        pivot_row, min_ratio = None, None
        for i in range(1, m+1):
            col_val = T[i][enter_col]
            if col_val > 0:
                ratio = T[i][-1] / col_val
                if pivot_row is None or ratio < min_ratio:
                    pivot_row, min_ratio = i, ratio
        if pivot_row is None:
            raise ValueError("Problema ilimitado")

        # ─── (IMPRESSÃO 2)  r e razão ─────────────────────────────
        print(f"  r  (sai)   : {base[pivot_row-1]}")
        print(f"  razão min  : {min_ratio}\n")

        # 3. pivoteamento (PASSO d)
        piv = T[pivot_row][enter_col]
        T[pivot_row] = [x / piv for x in T[pivot_row]]
        for i in range(m+1):
            if i == pivot_row:
                continue
            factor = T[i][enter_col]
            T[i] = [x - factor*y for x, y in zip(T[i], T[pivot_row])]

        # 4. atualiza base e imprime tableau
        base[pivot_row-1] = var[enter_col]
        print_tableau(it)

    else:
        raise RuntimeError("Limite de iterações atingido")

    # solução
    x = [Fraction(0) for _ in range(n)]
    for i, bv in enumerate(base):
        x[var.index(bv)] = T[i+1][-1]
    z = T[0][-1]
    return x, z


# Solução Gráfica

In [7]:
import numpy as np
import matplotlib.pyplot as plt

def plot_feasible_region(constraints,
                         x1_range=(0, 10),
                         x2_range=(0, 10),
                         resolution=400,
                         fill_region=True,
                         show_labels=True):
    """
    Desenha a região viável para um sistema de inequações lineares em x1 e x2.

    Parameters
    ----------
    constraints : list[dict]
        Cada dicionário descreve uma desigualdade do tipo
            a1 * x1 + a2 * x2  op  b
        Campos obrigatórios:
            'a1', 'a2', 'b' : coeficientes numéricos
            'op'            : '<=' ou '>='
        Campos opcionais:
            'color'         : cor da reta
            'label'         : texto exibido no gráfico
    x1_range, x2_range : tuple(float, float)
        Limites dos eixos x1 e x2.
    resolution : int
        Número de pontos da malha.
    fill_region : bool
        Se True, preenche a região viável.
    show_labels : bool
        Se True, escreve os rótulos das restrições.
    """

    # Geração da malha
    x1_vals = np.linspace(*x1_range, resolution)
    x2_vals = np.linspace(*x2_range, resolution)
    X1, X2 = np.meshgrid(x1_vals, x2_vals)

    feasible = np.ones_like(X1, dtype=bool)

    # Setup gráfico
    fig, ax = plt.subplots(figsize=(8, 6))
    default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    for idx, c in enumerate(constraints):
        a1, a2, b, op = c['a1'], c['a2'], c['b'], c['op']
        color = c.get('color', default_colors[idx % len(default_colors)])
        label = c.get('label', f'{a1}x₁ + {a2}x₂ {op} {b}')

        # Corrige \le e \ge para matplotlib
        label = label.replace(r'\le', r'\leq').replace(r'\ge', r'\geq')

        # Atualiza região viável
        expr = a1 * X1 + a2 * X2
        if op == '<=':
            feasible &= (expr <= b + 1e-9)
        elif op == '>=':
            feasible &= (expr >= b - 1e-9)
        else:
            raise ValueError("op deve ser '<=' ou '>='")

        # Plota linha de fronteira
        if np.isclose(a2, 0):  # linha vertical
            x_const = b / a1
            ax.axvline(x_const, color=color)
            if show_labels:
                ax.text(x_const + 0.1, 0.05*(x2_range[1]-x2_range[0]),
                        label, color=color, rotation=90, va='bottom')
        else:
            y_line = (b - a1 * x1_vals) / a2
            ax.plot(x1_vals, y_line, color=color)
            if show_labels:
                x_mid = (x1_range[0] + x1_range[1]) / 4
                y_mid = (b - a1 * x_mid) / a2
                ax.text(x_mid, y_mid + 0.3, label, color=color)

    # Condições x1 >= 0, x2 >= 0
    ax.axhline(0, color='black')
    ax.axvline(0, color='black')
    feasible &= (X1 >= 0) & (X2 >= 0)

    if fill_region:
        ax.contourf(X1, X2, feasible, levels=[0.5, 1], colors=['#D3D3D3'])

    # Ajustes finais
    ax.set_xlim(x1_range)
    ax.set_ylim(x2_range)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title('Região Viável')
    ax.grid(True)
    ax.legend()
    plt.show()

Exemplo simplex tabela

In [ ]:
A = [
    [2, 1, 1, 0, 0],   # 2x1 +  x2 + x3 = 8
    [1, 2, 0, 1, 0],   #  x1 + 2x2 + x4 = 7
    [0, 1, 0, 0, 1]    #       x2 + x5 = 3
]
b = [8, 7, 3]
c = [-1, -1, 0, 0, 0]   # min -x1 - x2

x_opt, z_opt = simplex_tableau_min_frac(A, b, c)